# Experiment 8 — Myocardium-Grounded Frequency Selection

**Goal**: Evaluate whether the trained LVSA U-Net consensus mask improves
frequency-offset selection over the pure Exp5 neural ranker.

**Ablation selectors**:
1. `neural_only`   — Exp5 LOOCV top-1 (baseline, Exp7)
2. `burden_only`   — argmin intra-myocardial banding_burden (new CV with real mask)
3. `fused`         — composite: 0.40·neural + 0.25·homogeneity + 0.30·(1-burden) + 0.05·VLM
4. `fused+vlm`     — same composite + rank-style VLM demoted to 0.10 weight

**GT**: Excel `1stcohortSAX (1).xlsx` → `image no` − 1 (0-indexed slice).

**Metrics**: Top-1 / Top-3 / within-±1-step / MAE (Hz) vs GT.

In [8]:
import warnings
warnings.filterwarnings('ignore')

import json
import csv
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─── Project root on path ────────────────────────────────────────────────────
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.WARNING)
print('Root:', ROOT)

Root: E:\CINE-Test1


## 0. Configuration

In [9]:
CHECKPOINT = 'E:/cardiotune/backend/ml/Cardiac_Multi_view_segmentation/checkpoints/Unet_LVSA_best.pkl'
DATA_ROOT  = 'Dr-Dataset'
EXCEL_PATH = '1stcohortSAX (1).xlsx'
EXP5_CSV   = 'exp5_outputs/exp5_loocv_per_patient.csv'
SEG_DEVICE = 'cpu'     # 'cuda' if GPU available
RUN_VLM    = False     # Set True to run real VLM (slow; ~20 min on 3050)
VLM_MODEL  = 'Qwen/Qwen2-VL-2B-Instruct'

## 1. Load Ground Truth

In [10]:
import pandas as pd

gt = {}  # patient_id -> {'gt_slice': int (0-idx), 'gt_hz': float}

# The Excel has a blank first row; auto-detect the real header row
# (same logic as freqscout_agent.tools.physics.PhysicsDescriptor)
raw = pd.read_excel(EXCEL_PATH, header=None, dtype=str)
header_row = 0
for i, row in raw.iterrows():
    if row.dropna().shape[0] >= 4:
        header_row = int(i)
        break

df_excel = pd.read_excel(EXCEL_PATH, header=header_row)
df_excel.columns = [str(c).strip() for c in df_excel.columns]
print('Columns found:', list(df_excel.columns))

def find_col(df, *keywords):
    for col in df.columns:
        cl = col.lower()
        if all(k.lower() in cl for k in keywords):
            return col
    return None

id_col  = find_col(df_excel, 'anonymized') or find_col(df_excel, 'patient')
ino_col = find_col(df_excel, 'image', 'no')
man_col = find_col(df_excel, 'manual')
print(f'id_col={id_col!r}, ino_col={ino_col!r}, man_col={man_col!r}')

def norm_pid(val):
    s = str(val).strip()
    if s.lower() in ('nan', 'none', ''):
        return None
    if not s.startswith('Patient-'):
        s = 'Patient-' + s.lstrip('Patient').lstrip('-')
    return s

for _, row in df_excel.iterrows():
    pid_raw = row.get(id_col)
    try:
        if pd.isna(pid_raw):
            continue
    except Exception:
        pass
    pid = norm_pid(pid_raw)
    if not pid:
        continue
    ino = row.get(ino_col)
    try:
        if pd.isna(ino):
            continue
    except Exception:
        continue
    gt_slice = int(float(ino)) - 1   # 1-indexed -> 0-indexed
    gt_hz_raw = row.get(man_col) if man_col else None
    try:
        gt_hz = float(gt_hz_raw) if gt_hz_raw is not None and not pd.isna(gt_hz_raw) else None
    except Exception:
        gt_hz = None
    gt[pid] = {'gt_slice': gt_slice, 'gt_hz': gt_hz}

print(f'\nGT loaded for {len(gt)} patients.')
print(pd.DataFrame(gt).T)

StopIteration: 

## 2. Load Exp5 LOOCV scores

In [ ]:
import ast

exp5 = {}  # patient_id -> numpy array of per-slice scores (1-indexed keys → 0-indexed)
df5  = pd.read_csv(EXP5_CSV)
for _, row in df5.iterrows():
    pid   = str(row['patient_id']).strip()
    raw   = row['slice_scores']
    try:
        parsed = json.loads(raw)
    except Exception:
        parsed = ast.literal_eval(raw)
    # Keys are 1-indexed strings → convert to 0-indexed array
    keys   = sorted(int(k) for k in parsed)
    scores = np.array([parsed[str(k)] for k in keys], dtype=float)
    exp5[pid] = scores  # already in order 1..N → index 0..N-1

print(f'Exp5 scores loaded for {len(exp5)} patients.')

## 3. Load all DICOM images

In [ ]:
import pydicom, re

def load_patient(folder: Path):
    dcm_files = sorted(folder.glob('*.dcm')) or sorted(folder.glob('*.DCM'))
    loaded = []
    for f in dcm_files:
        try:
            ds = pydicom.dcmread(str(f))
            m  = re.search(r'img(\d+)', f.name, re.IGNORECASE)
            sk = int(m.group(1)) if m else int(getattr(ds, 'InstanceNumber', 0) or 0)
            loaded.append((sk, ds))
        except Exception:
            pass
    loaded.sort(key=lambda x: x[0])
    images = []
    for _, ds in loaded:
        arr = ds.pixel_array.astype(np.float32)
        if arr.ndim > 2:
            arr = arr.mean(axis=0)
        if arr.max() > arr.min():
            arr = (arr - arr.min()) / (arr.max() - arr.min())
        images.append(arr)
    return images

# Discover patients
patient_dirs = sorted(
    (p.name, p) for p in Path(DATA_ROOT).rglob('Patient-*') if p.is_dir()
)
print(f'Found {len(patient_dirs)} patient directories.')

# Load images (may take ~30 s)
all_images = {}
for pid, folder in patient_dirs:
    try:
        imgs = load_patient(folder)
        all_images[pid] = imgs
    except Exception as e:
        print(f'  WARN {pid}: {e}')
        
print(f'Loaded images for {len(all_images)} patients.')

## 4. Run LVSA U-Net segmentation (consensus mask)

In [ ]:
from freqscout_agent.tools.seg_advchain.adapter import LVSASegAdapter

print(f'Loading LVSA U-Net from {Path(CHECKPOINT).name} …')
adapter = LVSASegAdapter(CHECKPOINT, device=SEG_DEVICE)

masks = {}   # patient_id -> H×W uint8 consensus mask
seg_stats = []

for pid, imgs in all_images.items():
    mask = adapter.segment_consensus(imgs)
    masks[pid] = mask
    myo_px = int((mask == 2).sum())
    lv_px  = int((mask == 3).sum())
    seg_stats.append({'patient_id': pid, 'myo_px': myo_px, 'lv_px': lv_px, 'n': len(imgs)})
    print(f'  {pid}: MYO={myo_px}px LV={lv_px}px ({len(imgs)} images)')

seg_df = pd.DataFrame(seg_stats)
print(f'\nMean MYO area: {seg_df["myo_px"].mean():.0f} px  |  Min: {seg_df["myo_px"].min()} px')

### 4a. Visual QA — inspect a few masks

In [ ]:
QA_PATIENTS = list(all_images.keys())[:6]  # first 6; change as needed

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
COLOURS = {1: [1.0, 0.8, 0.0], 2: [0.2, 0.9, 0.2], 3: [0.2, 0.4, 1.0]}

for ax, pid in zip(axes.flat, QA_PATIENTS):
    imgs = all_images[pid]
    mask = masks[pid]
    # Use mid-index image
    mid  = len(imgs) // 2
    grey = imgs[mid]
    if grey.max() > 1.0:
        grey = grey / grey.max()
    rgb  = np.stack([grey, grey, grey], axis=-1).copy()
    for label, col in COLOURS.items():
        region = mask == label
        for c, v in enumerate(col):
            rgb[region, c] = 0.55 * rgb[region, c] + 0.45 * v
    ax.imshow(rgb)
    myo_px = int((mask == 2).sum())
    ax.set_title(f'{pid}\nMYO={myo_px}px', fontsize=9)
    ax.axis('off')

patches = [
    mpatches.Patch(color=[1,0.8,0], label='RV'),
    mpatches.Patch(color=[0.2,0.9,0.2], label='MYO'),
    mpatches.Patch(color=[0.2,0.4,1], label='LV'),
]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=10)
plt.suptitle('LVSA U-Net Consensus Segmentation — Visual QA', fontsize=13)
plt.tight_layout()
plt.savefig('exp8_seg_qc_preview.png', dpi=120)
plt.show()
print('Saved: exp8_seg_qc_preview.png')

## 5. Compute per-offset intra-myocardial metrics

In [ ]:
from freqscout_agent.tools.segment import SegmentationResult
from freqscout_agent.tools.banding import BandingDetector
from freqscout_agent.tools.quality import QualityMetrics

STEP_HZ = 25.0

def make_dummy_seg(pid, mask, n_images):
    """Wrap a consensus mask in a SegmentationResult for the banding/quality tools."""
    from scipy import ndimage as ndi
    lv = mask == 3
    rows, cols = np.where(lv)
    centroid = (rows.mean(), cols.mean()) if rows.size else (mask.shape[0]/2, mask.shape[1]/2)
    H, W = mask.shape
    cy, cx = centroid
    myo = mask == 2
    r, c = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
    angle = np.degrees(np.arctan2(r - cy, c - cx)) % 360
    sectors = {
        'anterior': myo & ((angle >= 315) | (angle < 45)),
        'lateral':  myo & (angle >= 45)  & (angle < 135),
        'inferior': myo & (angle >= 135) & (angle < 225),
        'septal':   myo & (angle >= 225) & (angle < 315),
    }
    offsets = [(i - n_images // 2) * STEP_HZ for i in range(n_images)]
    return SegmentationResult(
        patient_id=pid, source_slice_idx=n_images//2,
        source_offset_hz=0.0, mask=mask, lv_centroid=centroid,
        sectors=sectors,
        provenance={'method': 'lvsa_unet_consensus'},
    ), offsets

bd = BandingDetector()
qm = QualityMetrics()

results = {}   # pid -> {'banding': list, 'quality': list, 'offsets': list}

for pid, imgs in all_images.items():
    if pid not in masks:
        continue
    seg, offsets = make_dummy_seg(pid, masks[pid], len(imgs))
    b_results = bd.run(imgs, offsets, seg)
    q_results = qm.run(imgs, offsets, seg)
    results[pid] = {'banding': b_results, 'quality': q_results, 'offsets': offsets}

print(f'Computed banding/quality metrics for {len(results)} patients.')

## 6. Ablation: 4 selectors vs GT

In [ ]:
from scipy.special import softmax as scipy_softmax

def softmax(x):
    x = np.array(x, dtype=float)
    e = np.exp(x - x.max())
    return e / e.sum()

W_NEURAL   = 0.40
W_HOMOG    = 0.25
W_NO_BURDEN= 0.30
W_VLM      = 0.05   # ablation: bump to 0.10 in fused+vlm

rows = []
ABLATIONS = ['neural_only', 'burden_only', 'fused', 'fused_vlm']

for pid, res in results.items():
    if pid not in gt or pid not in exp5:
        continue
    g    = gt[pid]['gt_slice']
    g_hz = gt[pid]['gt_hz']
    offs = res['offsets']
    n    = len(offs)
    
    # Neural scores (Exp5 LOOCV, may be shorter than n)
    nn_raw = exp5.get(pid, np.zeros(n))
    if len(nn_raw) < n:
        nn_raw = np.concatenate([nn_raw, np.full(n - len(nn_raw), nn_raw.min() - 1)])
    nn_raw = nn_raw[:n]
    nn_soft = softmax(nn_raw)   # probability per slice

    # CV: banding burden and homogeneity per slice
    burden  = np.array([b.banding_burden for b in res['banding']])
    homog   = np.array([q.homogeneity    for q in res['quality']])
    in_myo  = np.array([b.bands_in_myocardium for b in res['banding']])
    in_pool = np.array([b.bands_in_lv_pool    for b in res['banding']])

    # Composite score
    composite = (
        W_NEURAL    * nn_soft
        + W_HOMOG   * homog
        + W_NO_BURDEN * (1.0 - burden)
        + W_VLM     * 0.5   # VLM neutral prior (real VLM not run in this ablation)
    )
    # Myo-overlap penalty
    penalty = np.where(in_myo | in_pool, 0.25, 1.0)
    composite_penalised = composite * penalty

    row = {'patient_id': pid, 'gt_slice': g, 'gt_hz': g_hz}
    
    # 1. Neural only
    row['neural_pred'] = int(np.argmax(nn_raw))
    row['neural_top3'] = list(np.argsort(nn_raw)[::-1][:3])
    
    # 2. Burden only (argmin burden; reject myo-overlap first)
    burden_score = burden.copy()
    burden_score[in_myo | in_pool] += 1.0  # hard penalty
    row['burden_pred'] = int(np.argmin(burden_score))
    row['burden_top3'] = list(np.argsort(burden_score)[:3])
    
    # 3. Fused (neural + CV, no VLM)
    row['fused_pred'] = int(np.argmax(composite_penalised))
    row['fused_top3'] = list(np.argsort(composite_penalised)[::-1][:3])
    
    # 4. fused+vlm placeholder (same as fused until real VLM is run)
    row['fused_vlm_pred'] = row['fused_pred']
    row['fused_vlm_top3'] = row['fused_top3']
    
    # Offsets for MAE
    row['neural_hz']  = offs[row['neural_pred']]
    row['burden_hz']  = offs[row['burden_pred']]
    row['fused_hz']   = offs[row['fused_pred']]
    rows.append(row)

abl_df = pd.DataFrame(rows)
print(f'{len(abl_df)} patients in ablation.')

## 7. Compute and display metrics

In [ ]:
def metrics(df, pred_col, top3_col, hz_col):
    n = len(df)
    top1 = (df[pred_col] == df['gt_slice']).sum()
    top3 = df.apply(lambda r: r['gt_slice'] in r[top3_col], axis=1).sum()
    w1   = df.apply(
        lambda r: abs(r[hz_col] - (r['gt_hz'] or r[hz_col])) <= 26, axis=1
    ).sum()
    mae  = df.apply(
        lambda r: abs(r[hz_col] - (r['gt_hz'] or r[hz_col])), axis=1
    ).mean()
    return {
        'Top-1': f'{top1}/{n} ({top1/n:.1%})',
        'Top-3': f'{top3}/{n} ({top3/n:.1%})',
        '±1 step': f'{w1}/{n} ({w1/n:.1%})',
        'MAE (Hz)': f'{mae:.1f}',
    }

summary = {
    'Exp5 neural-only (Exp7 baseline)': metrics(abl_df, 'neural_pred', 'neural_top3', 'neural_hz'),
    'Burden-only (new CV, real mask)':  metrics(abl_df, 'burden_pred', 'burden_top3', 'burden_hz'),
    'Fused (neural + CV)':              metrics(abl_df, 'fused_pred',  'fused_top3',  'fused_hz'),
    'Fused + VLM (placeholder)':        metrics(abl_df, 'fused_vlm_pred', 'fused_vlm_top3', 'fused_hz'),
}

res_df = pd.DataFrame(summary).T
print('\n=== Experiment 8 Ablation Results ===')
print(res_df.to_string())
res_df.to_csv('exp8_ablation_results.csv')
print('\nSaved: exp8_ablation_results.csv')

## 8. Per-patient breakdown

In [ ]:
abl_df['neural_ok'] = abl_df['neural_pred'] == abl_df['gt_slice']
abl_df['burden_ok'] = abl_df['burden_pred'] == abl_df['gt_slice']
abl_df['fused_ok']  = abl_df['fused_pred']  == abl_df['gt_slice']
abl_df['neural_d']  = (abl_df['neural_pred'] - abl_df['gt_slice']).abs()
abl_df['burden_d']  = (abl_df['burden_pred'] - abl_df['gt_slice']).abs()
abl_df['fused_d']   = (abl_df['fused_pred']  - abl_df['gt_slice']).abs()
abl_df['fused_beats_neural'] = (~abl_df['neural_ok']) & abl_df['fused_ok']
abl_df['fused_loses_neural'] = abl_df['neural_ok'] & (~abl_df['fused_ok'])

display_cols = ['patient_id','gt_slice','neural_pred','burden_pred','fused_pred',
                'neural_ok','fused_ok','fused_d']
print(abl_df[display_cols].to_string(index=False))

beats = abl_df['fused_beats_neural'].sum()
loses = abl_df['fused_loses_neural'].sum()
print(f'\nFused beats neural: {beats} patients | Fused loses to neural: {loses}')

## 9. Visualise banding burden across offsets (2 example patients)

In [ ]:
EXAMPLE_PIDS = list(results.keys())[:2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, pid in zip(axes, EXAMPLE_PIDS):
    res = results[pid]
    offs = res['offsets']
    burden = [b.banding_burden for b in res['banding']]
    homog  = [q.homogeneity    for q in res['quality']]
    nn_s   = softmax(exp5.get(pid, np.zeros(len(offs)))[:len(offs)])
    
    ax.plot(offs, burden, 'r-o', label='Banding burden', markersize=5)
    ax.plot(offs, [1-h for h in homog], 'b--s', label='1 - Homogeneity', markersize=4)
    ax.plot(offs, 1 - nn_s, 'g-.^', label='1 - Neural score', markersize=4)
    
    g = gt.get(pid, {}).get('gt_slice')
    if g is not None and g < len(offs):
        ax.axvline(offs[g], color='k', linestyle=':', linewidth=2, label=f'GT ({offs[g]:+.0f} Hz)')
    
    ax.set_xlabel('Frequency offset (Hz)')
    ax.set_ylabel('Score (lower = better)')
    ax.set_title(pid)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Intra-myocardial banding burden vs neural score across frequency offsets')
plt.tight_layout()
plt.savefig('exp8_burden_profile.png', dpi=120)
plt.show()

## 10. Run full agent pipeline (optional — saves to agent_outputs_exp8/)

In [ ]:
# Uncomment to run the full agentic pipeline on all patients.
# This will use the LVSA segmenter + VLM (if RUN_VLM=True) + knowledge retrieval.
# Expected time: ~10-20 min on 3050 without VLM; ~60-90 min with VLM.

# from freqscout_agent.pipeline import FreqScoutPipeline
# pipeline = FreqScoutPipeline.from_config('freqscout_agent/configs/models.yaml')
# pipeline._output_dir_override = 'agent_outputs_exp8'
# reports = pipeline.run_all_patients(data_root=DATA_ROOT)
# print(f'Completed {len(reports)} patients.')

---
## Summary

The cell above (Section 7) prints the key ablation table.  Interpret it as:

- If **burden_only > neural_only** on Top-1: the real myocardium mask adds signal.
- If **fused > neural_only**: the combination beats Exp5 alone.
- If **fused ≈ neural_only**: the CV metrics are still too noisy (mask QA issues) or
  the neural ranker is already near the ceiling for this dataset.

Next steps after running this notebook:
1. Check the QA overlays in `exp8_seg_qc/` — flag any patient with MYO < 200 px.
2. Rerun with `segmentation.device: cuda` on the lab GPU for speed.
3. Set `RUN_VLM = True` with the lab GPU (7B model) for the real VLM ablation.
4. Push results to GitHub as `exp8_outputs/` (zip first — gitignored by default).